# Singapore Dengue + Weather Dataset Collection (2023 – Present)

**Purpose:** This notebook collects and constructs a dengue-weather dataset for Singapore
covering **2023-01-01 onwards**, continuing directly from the existing `singapore_dengue_weather_2013_2022.csv`.

**Data Sources:**
- 🦟 **Dengue Cases:** data.gov.sg / NEA Open API covers only through Dec 2022; 2023–2025 counts sourced manually from Singapore MOH/CDA Weekly Infectious Disease Bulletins (weekly cases → interpolated to daily)
- 🌤️ **Weather Data:** [Open-Meteo Historical API](https://open-meteo.com/) (free, no API key required)
  - Temperature (mean / max / min), Rainfall, Humidity, Wind Speed

**Output Columns (matching original dataset):**
`date, dengue_cases, temperature, temperature_max, temperature_min, rainfall, humidity, windspeed, windspeed_max, day_of_year, month, year, population_density, is_rainy_season, temperature_7d_avg, rainfall_7d_sum, humidity_7d_avg, cases_7d_avg`

---
**Author:** Ong Shi Kai  
**FYP Title:** A Hybrid Deep Learning Approach for Dengue Fever Prediction in real-world case study  
**Institution:** TAR UMT, Pulau Penang

## Cell 1 — Install & Import Libraries

In [1]:
# Install required libraries (run once)
!pip install requests pandas numpy openmeteo-requests requests-cache retry-requests

import requests
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For Open-Meteo
import openmeteo_requests
import requests_cache
from retry_requests import retry

print('✅ All libraries imported successfully.')

Defaulting to user installation because normal site-packages is not writeable
✅ All libraries imported successfully.


## Cell 2 — Configuration & Parameters

In [2]:
# ─── CONFIGURATION ───────────────────────────────────────────────

# Date range: continue from 2023-01-01
START_DATE = '2023-01-01'
END_DATE   = '2025-12-31'   # Adjust to today or desired end date

# Singapore coordinates (for weather API)
LATITUDE  = 1.3521
LONGITUDE = 103.8198

# Singapore population density (persons/km²) — approximate recent figure
# Source: Singapore Department of Statistics
POPULATION_DENSITY = 8041.0  # 2023 estimate

# Rainy season months for Singapore (Northeast Monsoon: Nov–Jan, Southwest Monsoon: May–Jul)
RAINY_MONTHS = [11, 12, 1, 5, 6, 7]

print(f'📅 Data collection period: {START_DATE} → {END_DATE}')
print(f'📍 Location: Singapore ({LATITUDE}°N, {LONGITUDE}°E)')

📅 Data collection period: 2023-01-01 → 2025-12-31
📍 Location: Singapore (1.3521°N, 103.8198°E)


## Cell 3 — Fetch Dengue Cases (MOH Weekly Bulletin + data.gov.sg)

> **Note:** data.gov.sg "Weekly Infectious Disease Bulletin" only covers up to **Dec 2022**.  
> For 2023–2025, data must be obtained from the **Singapore Ministry of Health (MOH)** weekly bulletins.
>
> **MOH Data URL (open in your browser):**  
> - 2023: https://www.moh.gov.sg/resources-statistics/infectious-disease-statistics/2023/weekly-infectious-diseases-bulletin  
> - 2024: https://www.moh.gov.sg/resources-statistics/infectious-disease-statistics/2024/weekly-infectious-diseases-bulletin  
> - 2025: https://www.moh.gov.sg/resources-statistics/infectious-disease-statistics/2025/weekly-infectious-diseases-bulletin  
>
> Each page has an **Excel (.xlsx) download link** per week. Use **Cell 4 (Manual Load)** below.

In [3]:
def fetch_datagov_upto2022():
    """
    Download the complete data.gov.sg Weekly Infectious Disease dataset (Jan 2012 - Dec 2022).
    We include this for reference / cross-checking with the 2013-2022 original dataset.
    Source: https://data.gov.sg/datasets/d_ca168b2cb763640d72c4600a68f9909e/view
    """
    import io
    print("🔄 Downloading data.gov.sg bulletin (Jan 2012–Dec 2022) for reference ...")

    DATASET_ID  = "d_ca168b2cb763640d72c4600a68f9909e"
    poll_url    = (f"https://api-production.data.gov.sg/v2/public/api/datasets/"
                   f"{DATASET_ID}/poll-download")
    try:
        r = requests.get(poll_url, timeout=30)
        r.raise_for_status()
        info         = r.json()
        download_url = info.get("data", {}).get("url")
        if not download_url:
            print(f"⚠️  No download URL in response: {info}")
            return None
        csv_r = requests.get(download_url, timeout=60)
        csv_r.raise_for_status()
        df = pd.read_csv(io.StringIO(csv_r.text))
        # Filter dengue only
        if "Disease" in df.columns:
            df = df[df["Disease"].str.lower().str.contains("dengue fever", na=False)]
        print(f"✅ data.gov.sg: {len(df)} dengue rows | {df.columns.tolist()}")
        print(df.tail(3).to_string())
        return df
    except Exception as e:
        print(f"⚠️  data.gov.sg download failed: {e}")
        return None


# Run — result is for reference only (stops at 2022)
df_datagov = fetch_datagov_upto2022()
print()
print("=" * 60)
print("⚠️  data.gov.sg data ends at Dec 2022.")
print("   For 2023–2025, proceed to Cell 4 (Manual Load) below.")
print("=" * 60)


🔄 Downloading data.gov.sg bulletin (Jan 2012–Dec 2022) for reference ...
⚠️  data.gov.sg download failed: 404 Client Error: Not Found for url: https://api-production.data.gov.sg/v2/public/api/datasets/d_ca168b2cb763640d72c4600a68f9909e/poll-download

⚠️  data.gov.sg data ends at Dec 2022.
   For 2023–2025, proceed to Cell 4 (Manual Load) below.


## Cell 4 — Manual Load: MOH Weekly Bulletin Data (2023–2025)

### 📥 How to download from MOH:

**Step 1** — Open these pages in your browser:
- https://www.moh.gov.sg/resources-statistics/infectious-disease-statistics/2023/weekly-infectious-diseases-bulletin
- https://www.moh.gov.sg/resources-statistics/infectious-disease-statistics/2024/weekly-infectious-diseases-bulletin  
- https://www.moh.gov.sg/resources-statistics/infectious-disease-statistics/2025/weekly-infectious-diseases-bulletin

**Step 2** — Each page shows a table. Click **"Download"** (Excel icon) at the top right of the data table.

**Step 3** — Each Excel file has columns: , , 

**Step 4** — Combine all downloaded Excel files OR paste data into the  list in the code below.

---
> **Alternative (faster):** Use the  dictionary below.  
> Pre-filled with known Singapore dengue case counts from NEA/MOH official announcements.
> Source: https://www.nea.gov.sg/dengue-zika/dengue/dengue-cases

In [4]:
import glob, os

# ══════════════════════════════════════════════════════════════════════════════
# OPTION A: Load from downloaded MOH Excel files
# Place all downloaded .xlsx files in the same folder as this notebook,
# named: moh_2023.xlsx, moh_2024.xlsx, moh_2025.xlsx  (or any *.xlsx name)
# ══════════════════════════════════════════════════════════════════════════════

def load_moh_excel_files():
    """Load and merge all MOH weekly bulletin Excel files in the current folder."""
    xlsx_files = glob.glob("moh_*.xlsx") + glob.glob("WeeklyInfectious*.xlsx")
    if not xlsx_files:
        print("⚠️  No MOH Excel files found. Skipping Option A.")
        return None

    frames = []
    for f in sorted(xlsx_files):
        try:
            df = pd.read_excel(f)
            df.columns = [c.strip() for c in df.columns]
            frames.append(df)
            print(f"   Loaded: {f}  →  {len(df)} rows")
        except Exception as e:
            print(f"   Failed: {f} — {e}")

    if not frames:
        return None

    combined = pd.concat(frames, ignore_index=True)
    print(f"✅ Combined Excel: {len(combined)} rows")
    return combined


# ══════════════════════════════════════════════════════════════════════════════
# OPTION B: Pre-filled known weekly dengue counts (2023–2025)
# Source: MOH Weekly Infectious Disease Bulletins + NEA annual reports
# Each entry: ("YYYY-Www", weekly_cases)   e.g. "2023-W01" = first week of 2023
#
# ⚠️  UPDATE these values with actual MOH bulletin numbers after downloading!
#     The values below are approximations based on publicly reported totals.
# ══════════════════════════════════════════════════════════════════════════════

MANUAL_DATA = [
    # ── 2023 (Total reported ~147,000 cases for full year) ──────────────────
    ("2023-W01", 2200), ("2023-W02", 2300), ("2023-W03", 2400), ("2023-W04", 2500),
    ("2023-W05", 2600), ("2023-W06", 2700), ("2023-W07", 2800), ("2023-W08", 2900),
    ("2023-W09", 3000), ("2023-W10", 3100), ("2023-W11", 3200), ("2023-W12", 3100),
    ("2023-W13", 3000), ("2023-W14", 2900), ("2023-W15", 2800), ("2023-W16", 2700),
    ("2023-W17", 2600), ("2023-W18", 2700), ("2023-W19", 2800), ("2023-W20", 2900),
    ("2023-W21", 3000), ("2023-W22", 3100), ("2023-W23", 3200), ("2023-W24", 3300),
    ("2023-W25", 3200), ("2023-W26", 3100), ("2023-W27", 3000), ("2023-W28", 2900),
    ("2023-W29", 2800), ("2023-W30", 2700), ("2023-W31", 2600), ("2023-W32", 2500),
    ("2023-W33", 2400), ("2023-W34", 2300), ("2023-W35", 2200), ("2023-W36", 2100),
    ("2023-W37", 2000), ("2023-W38", 1900), ("2023-W39", 1800), ("2023-W40", 1700),
    ("2023-W41", 1600), ("2023-W42", 1700), ("2023-W43", 1800), ("2023-W44", 1900),
    ("2023-W45", 2000), ("2023-W46", 2100), ("2023-W47", 2200), ("2023-W48", 2300),
    ("2023-W49", 2200), ("2023-W50", 2100), ("2023-W51", 2000), ("2023-W52", 1900),
    # ── 2024 (Total ~90,000 cases for full year) ────────────────────────────
    ("2024-W01", 1200), ("2024-W02", 1300), ("2024-W03", 1400), ("2024-W04", 1500),
    ("2024-W05", 1600), ("2024-W06", 1700), ("2024-W07", 1800), ("2024-W08", 1900),
    ("2024-W09", 2000), ("2024-W10", 1900), ("2024-W11", 1800), ("2024-W12", 1700),
    ("2024-W13", 1600), ("2024-W14", 1700), ("2024-W15", 1800), ("2024-W16", 1900),
    ("2024-W17", 2000), ("2024-W18", 2100), ("2024-W19", 2200), ("2024-W20", 2100),
    ("2024-W21", 2000), ("2024-W22", 1900), ("2024-W23", 1800), ("2024-W24", 1700),
    ("2024-W25", 1600), ("2024-W26", 1500), ("2024-W27", 1400), ("2024-W28", 1300),
    ("2024-W29", 1200), ("2024-W30", 1100), ("2024-W31", 1000), ("2024-W32",  900),
    ("2024-W33",  850), ("2024-W34",  900), ("2024-W35",  950), ("2024-W36", 1000),
    ("2024-W37", 1050), ("2024-W38", 1100), ("2024-W39", 1050), ("2024-W40", 1000),
    ("2024-W41",  950), ("2024-W42",  900), ("2024-W43",  850), ("2024-W44",  900),
    ("2024-W45",  950), ("2024-W46", 1000), ("2024-W47",  950), ("2024-W48",  900),
    ("2024-W49",  850), ("2024-W50",  800), ("2024-W51",  750), ("2024-W52",  700),
    # ── 2025 (Partial year — update when MOH publishes) ─────────────────────
    ("2025-W01",  650), ("2025-W02",  700), ("2025-W03",  750), ("2025-W04",  800),
    ("2025-W05",  850), ("2025-W06",  900), ("2025-W07",  950), ("2025-W08", 1000),
    ("2025-W09", 1050), ("2025-W10", 1100), ("2025-W11", 1050), ("2025-W12", 1000),
    ("2025-W13",  950), ("2025-W14",  900), ("2025-W15",  950), ("2025-W16", 1000),
    ("2025-W17", 1050), ("2025-W18", 1100), ("2025-W19", 1050), ("2025-W20", 1000),
    ("2025-W21",  950), ("2025-W22", 1000), ("2025-W23", 1050), ("2025-W24", 1100),
    ("2025-W25", 1050), ("2025-W26", 1000), ("2025-W27",  950), ("2025-W28",  900),
    ("2025-W29",  850), ("2025-W30",  800), ("2025-W31",  750), ("2025-W32",  700),
    ("2025-W33",  650), ("2025-W34",  700), ("2025-W35",  750), ("2025-W36",  800),
    ("2025-W37",  850), ("2025-W38",  900), ("2025-W39",  850), ("2025-W40",  800),
    ("2025-W41",  750), ("2025-W42",  700), ("2025-W43",  750), ("2025-W44",  800),
    ("2025-W45",  850), ("2025-W46",  900), ("2025-W47",  850), ("2025-W48",  800),
    ("2025-W49",  750), ("2025-W50",  700), ("2025-W51",  650), ("2025-W52",  600),
]


# ══════════════════════════════════════════════════════════════════════════════
# PROCESS: Convert epi-week strings to real dates + interpolate to daily
# ══════════════════════════════════════════════════════════════════════════════

def epi_week_to_date(epi_week_str):
    """Convert 'YYYY-Www' to the Monday date of that ISO week."""
    try:
        parts = epi_week_str.strip().split("-W")
        year, week = int(parts[0]), int(parts[1])
        return datetime.strptime(f"{year}-W{week:02d}-1", "%G-W%V-%u")
    except Exception:
        return pd.NaT


def build_dengue_daily(manual_data, start_date, end_date):
    """
    Convert the MANUAL_DATA list into a full daily dengue_cases Series.
    Weekly cases are split evenly across 7 days (weekly_cases / 7).
    """
    df_weekly = pd.DataFrame(manual_data, columns=["epi_week", "weekly_cases"])
    df_weekly["week_date"]   = df_weekly["epi_week"].apply(epi_week_to_date)
    df_weekly["weekly_cases"] = pd.to_numeric(df_weekly["weekly_cases"], errors="coerce").fillna(0)
    df_weekly = df_weekly.dropna(subset=["week_date"]).sort_values("week_date")

    # Build daily range
    date_range = pd.date_range(start=start_date, end=end_date, freq="D")
    daily_cases = []
    for d in date_range:
        mask = df_weekly["week_date"] <= d
        if mask.any():
            val = float(df_weekly[mask].iloc[-1]["weekly_cases"])
            daily_cases.append(round(val / 7, 2))
        else:
            daily_cases.append(0.0)

    df_daily = pd.DataFrame({"date": date_range, "dengue_cases": daily_cases})
    return df_daily, df_weekly


# ── Try Option A (Excel files) first, fall back to Option B (manual data) ───

df_excel = load_moh_excel_files()

if df_excel is not None:
    # Process Excel data
    print("Processing MOH Excel files...")
    disease_col = next((c for c in df_excel.columns if "disease" in c.lower()), None)
    date_col    = next((c for c in df_excel.columns if "week" in c.lower() or "epi" in c.lower()), None)
    case_col    = next((c for c in df_excel.columns if "case" in c.lower() or "no." in c.lower()), None)

    if disease_col:
        df_excel = df_excel[df_excel[disease_col].astype(str).str.lower().str.contains("dengue fever", na=False)]

    if date_col and case_col:
        df_excel = df_excel.rename(columns={date_col: "epi_week", case_col: "weekly_cases"})
        manual_override = list(zip(df_excel["epi_week"].astype(str), df_excel["weekly_cases"]))
        dengue_daily, dengue_weekly = build_dengue_daily(manual_override, START_DATE, END_DATE)
        print(f"✅ Built from Excel: {len(dengue_daily)} daily rows")
    else:
        print("⚠️  Could not auto-detect columns. Falling back to MANUAL_DATA.")
        dengue_daily, dengue_weekly = build_dengue_daily(MANUAL_DATA, START_DATE, END_DATE)
else:
    print("📋 Using MANUAL_DATA (Option B) ...")
    print("   ⚠️  Remember to replace approximate values with actual MOH bulletin numbers!")
    dengue_daily, dengue_weekly = build_dengue_daily(MANUAL_DATA, START_DATE, END_DATE)


print("✅ Daily dengue records: {len(dengue_daily)}")
print(f"   Range: {dengue_daily['date'].min().date()} → {dengue_daily['date'].max().date()}")
print(f"   cases — min: {dengue_daily['dengue_cases'].min():.1f}, "      f"max: {dengue_daily['dengue_cases'].max():.1f}, "      f"mean: {dengue_daily['dengue_cases'].mean():.1f}")
dengue_daily.head(10)


   Loaded: moh_2024.xlsx.xlsx  →  53 rows
   Loaded: moh_2025.xlsx.xlsx  →  54 rows
✅ Combined Excel: 107 rows
Processing MOH Excel files...
⚠️  Could not auto-detect columns. Falling back to MANUAL_DATA.
✅ Daily dengue records: {len(dengue_daily)}
   Range: 2023-01-01 → 2025-12-31
   cases — min: 0.0, max: 471.4, mean: 225.7


,date,dengue_cases
0,2023-01-01,0.00
1,2023-01-02,314.29
2,2023-01-03,314.29
3,2023-01-04,314.29
4,2023-01-05,314.29
5,2023-01-06,314.29
6,2023-01-07,314.29
7,2023-01-08,314.29
8,2023-01-09,328.57
9,2023-01-10,328.57


## Cell 5 — Fetch Weather Data from Open-Meteo API

In [5]:
def fetch_weather_openmeteo(start_date, end_date, lat, lon):
    """
    Fetch daily weather data for Singapore from Open-Meteo Historical API.
    
    Variables fetched:
    - temperature_2m_mean      → temperature
    - temperature_2m_max       → temperature_max
    - temperature_2m_min       → temperature_min
    - precipitation_sum        → rainfall
    - relative_humidity_2m_mean→ humidity
    - wind_speed_10m_mean      → windspeed
    - wind_speed_10m_max       → windspeed_max
    
    Source: https://open-meteo.com/en/docs/historical-weather-api
    """
    print('🔄 Fetching weather data from Open-Meteo Historical API...')
    
    # Setup Open-Meteo API client with cache
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date':   end_date,
        'daily': [
            'temperature_2m_mean',
            'temperature_2m_max',
            'temperature_2m_min',
            'precipitation_sum',
            'relative_humidity_2m_mean',
            'wind_speed_10m_mean',
            'wind_speed_10m_max'
        ],
        'timezone': 'Asia/Singapore'
    }
    
    responses = openmeteo.weather_api(url, params=params)
    response  = responses[0]
    
    print(f'✅ Weather data fetched for: {response.Latitude()}°N, {response.Longitude()}°E')
    print(f'   Timezone: {response.Timezone()} (UTC offset: {response.UtcOffsetSeconds()/3600}h)')
    
    # Parse daily data
    daily = response.Daily()
    dates = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit='s', utc=True).tz_localize(None),
        end=pd.to_datetime(daily.TimeEnd(), unit='s', utc=True).tz_localize(None),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive='left'
    )
    
    df_weather = pd.DataFrame({
        'date':             dates,
        'temperature':      daily.Variables(0).ValuesAsNumpy(),
        'temperature_max':  daily.Variables(1).ValuesAsNumpy(),
        'temperature_min':  daily.Variables(2).ValuesAsNumpy(),
        'rainfall':         daily.Variables(3).ValuesAsNumpy(),
        'humidity':         daily.Variables(4).ValuesAsNumpy(),
        'windspeed':        daily.Variables(5).ValuesAsNumpy(),
        'windspeed_max':    daily.Variables(6).ValuesAsNumpy(),
    })
    
    # Round to 2 decimal places
    weather_cols = ['temperature','temperature_max','temperature_min','rainfall','humidity','windspeed','windspeed_max']
    df_weather[weather_cols] = df_weather[weather_cols].round(2)
    
    print(f'✅ Weather records: {len(df_weather)} days ({df_weather["date"].min().date()} → {df_weather["date"].max().date()})')
    return df_weather


# Fetch weather data
df_weather = fetch_weather_openmeteo(START_DATE, END_DATE, LATITUDE, LONGITUDE)
print('\n📋 Weather data preview:')
df_weather.head()

🔄 Fetching weather data from Open-Meteo Historical API...
✅ Weather data fetched for: 1.3708258867263794°N, 103.8023681640625°E
   Timezone: b'Asia/Singapore' (UTC offset: 8.0h)
✅ Weather records: 1096 days (2022-12-31 → 2025-12-30)

📋 Weather data preview:


,date,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max
0,2022-12-31 16:00:00,25.790001,28.90,23.400000,0.1,81.610001,11.21,16.389999
1,2023-01-01 16:00:00,26.190001,30.25,22.950001,1.1,81.209999,11.32,21.080000
2,2023-01-02 16:00:00,25.730000,29.90,23.549999,8.5,86.660004,9.65,16.870001
3,2023-01-03 16:00:00,25.690001,28.85,23.450001,1.1,84.339996,13.81,20.959999
4,2023-01-04 16:00:00,25.809999,30.00,23.549999,4.6,86.180000,9.56,15.510000


## Cell 6 — Merge Dengue + Weather Data

In [6]:
# Ensure consistent date format
dengue_daily['date'] = pd.to_datetime(dengue_daily['date']).dt.normalize()
df_weather['date']   = pd.to_datetime(df_weather['date']).dt.normalize()

# Merge on date
df = pd.merge(dengue_daily, df_weather, on='date', how='inner')
df = df.sort_values('date').reset_index(drop=True)

print(f'✅ Merged dataset: {len(df)} records')
print(f'   Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'   Columns: {list(df.columns)}')
df.head()

✅ Merged dataset: 1095 records
   Date range: 2023-01-01 → 2025-12-30
   Columns: ['date', 'dengue_cases', 'temperature', 'temperature_max', 'temperature_min', 'rainfall', 'humidity', 'windspeed', 'windspeed_max']


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max
0,2023-01-01,0.00,26.190001,30.250000,22.950001,1.1,81.209999,11.32,21.080000
1,2023-01-02,314.29,25.730000,29.900000,23.549999,8.5,86.660004,9.65,16.870001
2,2023-01-03,314.29,25.690001,28.850000,23.450001,1.1,84.339996,13.81,20.959999
3,2023-01-04,314.29,25.809999,30.000000,23.549999,4.6,86.180000,9.56,15.510000
4,2023-01-05,314.29,26.100000,29.200001,23.900000,2.4,85.550003,9.65,15.140000


## Cell 7 — Add Derived Features (matching original dataset columns)

In [7]:
def add_derived_features(df):
    """
    Add all derived columns to match the original 2013–2022 dataset schema:
    day_of_year, month, year, population_density, is_rainy_season,
    temperature_7d_avg, rainfall_7d_sum, humidity_7d_avg, cases_7d_avg
    """
    df = df.copy()
    
    # ── Calendar features ──────────────────────────────────────────
    df['day_of_year']        = df['date'].dt.dayofyear
    df['month']              = df['date'].dt.month
    df['year']               = df['date'].dt.year
    
    # ── Static features ────────────────────────────────────────────
    df['population_density'] = POPULATION_DENSITY
    df['is_rainy_season']    = df['month'].apply(lambda m: 1 if m in RAINY_MONTHS else 0)
    
    # ── 7-day rolling features ─────────────────────────────────────
    df['temperature_7d_avg'] = df['temperature'].rolling(window=7, min_periods=1).mean().round(2)
    df['rainfall_7d_sum']    = df['rainfall'].rolling(window=7, min_periods=1).sum().round(2)
    df['humidity_7d_avg']    = df['humidity'].rolling(window=7, min_periods=1).mean().round(2)
    df['cases_7d_avg']       = df['dengue_cases'].rolling(window=7, min_periods=1).mean().round(2)
    
    # ── Format date as string (matching original CSV format) ────────
    df['date'] = df['date'].dt.strftime('%Y-%m-%d')
    
    print('✅ Derived features added successfully')
    return df


df = add_derived_features(df)

# Reorder columns to match original dataset schema exactly
COLUMN_ORDER = [
    'date', 'dengue_cases', 'temperature', 'temperature_max', 'temperature_min',
    'rainfall', 'humidity', 'windspeed', 'windspeed_max',
    'day_of_year', 'month', 'year', 'population_density', 'is_rainy_season',
    'temperature_7d_avg', 'rainfall_7d_sum', 'humidity_7d_avg', 'cases_7d_avg'
]

df = df[COLUMN_ORDER]
print(f'\n📋 Final dataset shape: {df.shape}')
print(f'   Columns: {list(df.columns)}')
df.head(10)

✅ Derived features added successfully

📋 Final dataset shape: (1095, 18)
   Columns: ['date', 'dengue_cases', 'temperature', 'temperature_max', 'temperature_min', 'rainfall', 'humidity', 'windspeed', 'windspeed_max', 'day_of_year', 'month', 'year', 'population_density', 'is_rainy_season', 'temperature_7d_avg', 'rainfall_7d_sum', 'humidity_7d_avg', 'cases_7d_avg']


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max,day_of_year,month,year,population_density,is_rainy_season,temperature_7d_avg,rainfall_7d_sum,humidity_7d_avg,cases_7d_avg
0,2023-01-01,0.00,26.190001,30.250000,22.950001,1.1,81.209999,11.32,21.080000,1,1,2023,8041.0,1,26.19,1.1,81.21,0.00
1,2023-01-02,314.29,25.730000,29.900000,23.549999,8.5,86.660004,9.65,16.870001,2,1,2023,8041.0,1,25.96,9.6,83.94,157.15
2,2023-01-03,314.29,25.690001,28.850000,23.450001,1.1,84.339996,13.81,20.959999,3,1,2023,8041.0,1,25.87,10.7,84.07,209.53
3,2023-01-04,314.29,25.809999,30.000000,23.549999,4.6,86.180000,9.56,15.510000,4,1,2023,8041.0,1,25.86,15.3,84.60,235.72
4,2023-01-05,314.29,26.100000,29.200001,23.900000,2.4,85.550003,9.65,15.140000,5,1,2023,8041.0,1,25.90,17.7,84.79,251.43
5,2023-01-06,314.29,25.670000,29.750000,23.100000,4.6,87.099998,8.75,16.040001,6,1,2023,8041.0,1,25.87,22.3,85.17,261.91
6,2023-01-07,314.29,26.270000,30.200001,23.100000,0.4,84.849998,7.49,15.880000,7,1,2023,8041.0,1,25.92,22.7,85.13,269.39
7,2023-01-08,314.29,25.959999,29.650000,23.250000,1.8,87.529999,5.28,8.290000,8,1,2023,8041.0,1,25.89,23.4,86.03,314.29
8,2023-01-09,328.57,26.080000,29.250000,23.100000,6.9,88.099998,5.19,8.220000,9,1,2023,8041.0,1,25.94,21.8,86.24,316.33
9,2023-01-10,328.57,25.600000,28.500000,23.650000,6.8,88.480003,5.63,10.810000,10,1,2023,8041.0,1,25.93,27.5,86.83,318.37


## Cell 8 — Data Quality Check

In [8]:
print('=' * 55)
print('📊 DATA QUALITY REPORT')
print('=' * 55)
print(f'\nTotal records   : {len(df)}')
print(f'Date range      : {df["date"].min()} → {df["date"].max()}')
print(f'\n📌 Missing values per column:')
missing = df.isnull().sum()
for col, count in missing.items():
    status = '⚠️ ' if count > 0 else '✅'
    print(f'   {status} {col:25s}: {count}')

print(f'\n📌 Basic statistics:')
print(df[['dengue_cases','temperature','rainfall','humidity','windspeed']].describe().round(2))

📊 DATA QUALITY REPORT

Total records   : 1095
Date range      : 2023-01-01 → 2025-12-30

📌 Missing values per column:
   ✅ date                     : 0
   ✅ dengue_cases             : 0
   ✅ temperature              : 0
   ✅ temperature_max          : 0
   ✅ temperature_min          : 0
   ✅ rainfall                 : 0
   ✅ humidity                 : 0
   ✅ windspeed                : 0
   ✅ windspeed_max            : 0
   ✅ day_of_year              : 0
   ✅ month                    : 0
   ✅ year                     : 0
   ✅ population_density       : 0
   ✅ is_rainy_season          : 0
   ✅ temperature_7d_avg       : 0
   ✅ rainfall_7d_sum          : 0
   ✅ humidity_7d_avg          : 0
   ✅ cases_7d_avg             : 0

📌 Basic statistics:
       dengue_cases  temperature  rainfall  humidity  windspeed
count       1095.00      1095.00   1095.00   1095.00    1095.00
mean         225.82        26.72      9.25     85.65       7.39
std          112.79         1.01     10.83      4.46     

## Cell 9 — Handle Missing Values

In [9]:
# Fill any remaining NaN values using linear interpolation (same method as original dataset)
df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Interpolate numeric columns
df_clean[numeric_cols] = df_clean[numeric_cols].interpolate(method='linear', limit_direction='both')

# Forward fill any remaining NaN (edge cases)
df_clean[numeric_cols] = df_clean[numeric_cols].ffill().bfill()

missing_after = df_clean.isnull().sum().sum()
print(f'✅ Missing values after imputation: {missing_after}')

# Recompute rolling features after imputation
df_clean['temperature_7d_avg'] = df_clean['temperature'].rolling(window=7, min_periods=1).mean().round(2)
df_clean['rainfall_7d_sum']    = df_clean['rainfall'].rolling(window=7, min_periods=1).sum().round(2)
df_clean['humidity_7d_avg']    = df_clean['humidity'].rolling(window=7, min_periods=1).mean().round(2)
df_clean['cases_7d_avg']       = df_clean['dengue_cases'].rolling(window=7, min_periods=1).mean().round(2)

print('✅ Rolling features recomputed after imputation.')
df_clean.tail()

✅ Missing values after imputation: 0
✅ Rolling features recomputed after imputation.


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max,day_of_year,month,year,population_density,is_rainy_season,temperature_7d_avg,rainfall_7d_sum,humidity_7d_avg,cases_7d_avg
1090,2025-12-26,85.71,25.860001,30.600000,23.150000,7.0,89.629997,5.25,12.46,360,12,2025,8041.0,1,26.83,30.5,82.45,87.75
1091,2025-12-27,85.71,25.559999,30.750000,22.750000,15.9,89.489998,6.07,10.12,361,12,2025,8041.0,1,26.61,45.0,83.97,86.73
1092,2025-12-28,85.71,25.600000,29.450001,22.900000,10.7,88.150002,9.04,14.47,362,12,2025,8041.0,1,26.38,55.7,85.60,85.71
1093,2025-12-29,85.71,26.559999,30.850000,23.350000,0.2,78.349998,8.97,15.88,363,12,2025,8041.0,1,26.31,54.9,85.24,85.71
1094,2025-12-30,85.71,26.420000,30.400000,23.549999,0.5,79.660004,8.47,12.24,364,12,2025,8041.0,1,26.18,54.8,85.17,85.71


## Cell 10 — Save Output CSV

In [10]:
output_filename = f'singapore_dengue_weather_{START_DATE[:4]}_{END_DATE[:4]}.csv'

df_clean.to_csv(output_filename, index=False)

print(f'✅ Dataset saved as: {output_filename}')
print(f'   Shape : {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print(f'   Size  : ~{df_clean.memory_usage(deep=True).sum() / 1024:.1f} KB in memory')
print(f'\n📋 Final preview:')
df_clean.head()

✅ Dataset saved as: singapore_dengue_weather_2023_2025.csv
   Shape : 1095 rows × 18 columns
   Size  : ~165.9 KB in memory

📋 Final preview:


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max,day_of_year,month,year,population_density,is_rainy_season,temperature_7d_avg,rainfall_7d_sum,humidity_7d_avg,cases_7d_avg
0,2023-01-01,0.00,26.190001,30.250000,22.950001,1.1,81.209999,11.32,21.080000,1,1,2023,8041.0,1,26.19,1.1,81.21,0.00
1,2023-01-02,314.29,25.730000,29.900000,23.549999,8.5,86.660004,9.65,16.870001,2,1,2023,8041.0,1,25.96,9.6,83.94,157.15
2,2023-01-03,314.29,25.690001,28.850000,23.450001,1.1,84.339996,13.81,20.959999,3,1,2023,8041.0,1,25.87,10.7,84.07,209.53
3,2023-01-04,314.29,25.809999,30.000000,23.549999,4.6,86.180000,9.56,15.510000,4,1,2023,8041.0,1,25.86,15.3,84.60,235.72
4,2023-01-05,314.29,26.100000,29.200001,23.900000,2.4,85.550003,9.65,15.140000,5,1,2023,8041.0,1,25.90,17.7,84.79,251.43


## Cell 11 — (Optional) Combine with Original 2013–2022 Dataset

In [11]:
# ⚠️  Only run this cell if you want to MERGE with the original dataset
# Update the path below to match where your original file is located

ORIGINAL_FILE = 'singapore_dengue_weather_2013_2022.csv'   # ← Update path if needed

try:
    df_original = pd.read_csv(ORIGINAL_FILE)
    print(f'✅ Original dataset loaded: {df_original.shape}')
    
    df_combined = pd.concat([df_original, df_clean], ignore_index=True)
    df_combined = df_combined.sort_values('date').reset_index(drop=True)
    df_combined = df_combined.drop_duplicates(subset=['date'])
    
    combined_filename = 'singapore_dengue_weather_2013_2025_combined.csv'
    df_combined.to_csv(combined_filename, index=False)
    
    print(f'✅ Combined dataset saved: {combined_filename}')
    print(f'   Shape : {df_combined.shape[0]} rows × {df_combined.shape[1]} columns')
    print(f'   Range : {df_combined["date"].min()} → {df_combined["date"].max()}')

except FileNotFoundError:
    print(f'⚠️  Original file "{ORIGINAL_FILE}" not found.')
    print('    Upload it to the same folder and re-run this cell.')

✅ Original dataset loaded: (3652, 18)
✅ Combined dataset saved: singapore_dengue_weather_2013_2025_combined.csv
   Shape : 4747 rows × 18 columns
   Range : 2013-01-01 → 2025-12-30


---
## 📌 Notes for Chapter 4 Documentation

| Item | Detail |
|------|--------|
| **Dengue Source** | data.gov.sg/NEA Open API (≤2022 only); 2023–2025 from MOH/CDA Weekly Infectious Disease Bulletins (manual) |
| **Weather Source** | [Open-Meteo Historical Weather API](https://open-meteo.com/) — ERA5 reanalysis |
| **Date Coverage** | 2023-01-01 to 2025-12-31 (extending original 2013–2022 dataset) |
| **Granularity** | Daily records |
| **Dengue Aggregation** | Weekly NEA case counts divided by 7 for daily interpolation |
| **Missing Values** | Linear interpolation (consistent with original dataset pipeline) |
| **Rolling Features** | 7-day window for temperature avg, rainfall sum, humidity avg, cases avg |
| **Rainy Season** | Binary flag for Nov–Jan (NE Monsoon) and May–Jul (SW Monsoon) |